# exp_10b — Experiment 10B: Denoising Autoencoder for MNIST

Source-based notebook converted from the uploaded experiment code document.


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

In [ ]:
(X_train, _), (X_test, _) = mnist.load_data()

print(X_train.shape)
print(X_test.shape)

In [ ]:
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

In [ ]:
X_train = X_train.reshape((-1, 784))
X_test = X_test.reshape((-1, 784))

print(X_train.shape)
print(X_test.shape)

In [ ]:
import numpy as np

noise_factor = 0.5

X_train_noisy = X_train + noise_factor * np.random.normal(
    loc=0.0,
    scale=1.0,
    size=X_train.shape
)

X_test_noisy = X_test + noise_factor * np.random.normal(
    loc=0.0,
    scale=1.0,
    size=X_test.shape
)

# Keep pixel values between 0 and 1
X_train_noisy = np.clip(X_train_noisy, 0., 1.)
X_test_noisy = np.clip(X_test_noisy, 0., 1.)

In [ ]:
plt.figure(figsize=(15,3))

for i in range(10):
    ax = plt.subplot(1,10,i+1)
    plt.imshow(X_test_noisy[i].reshape(28,28), cmap="gray")
    plt.axis("off")

plt.show()

In [ ]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

encoding_dim = 32

input_img = Input(shape=(784,))

# Encoder
x = Dense(128, activation="relu")(input_img)
x = Dense(64, activation="relu")(x)
encoded = Dense(encoding_dim, activation="relu")(x)

# Decoder
x = Dense(64, activation="relu")(encoded)
x = Dense(128, activation="relu")(x)
decoded = Dense(784, activation="sigmoid")(x)

dae = Model(input_img, decoded)

In [ ]:
dae.compile(
    optimizer="adam",
    loss="binary_crossentropy"
)

dae.summary()

In [ ]:
history_dae = dae.fit(
    X_train_noisy,
    X_train,
    epochs=10,
    batch_size=256,
    shuffle=True,
    validation_data=(X_test_noisy, X_test)
)

In [ ]:
decoded_imgs = dae.predict(X_test_noisy)

In [ ]:
n = 10

plt.figure(figsize=(20,6))

for i in range(n):

    # Original Image
    ax = plt.subplot(3,n,i+1)
    plt.imshow(X_test[i].reshape(28,28), cmap="gray")
    plt.title("Original")
    plt.axis("off")

    # Noisy Image
    ax = plt.subplot(3,n,i+1+n)
    plt.imshow(X_test_noisy[i].reshape(28,28), cmap="gray")
    plt.title("Noisy")
    plt.axis("off")

    # Denoised Image
    ax = plt.subplot(3,n,i+1+2*n)
    plt.imshow(decoded_imgs[i].reshape(28,28), cmap="gray")
    plt.title("Denoised")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_dae.history["loss"], label="Training Loss")
plt.plot(history_dae.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Denoising Autoencoder Training")
plt.legend()

plt.show()